# Phase 1 RunPod Ablation — Qwen 0.5B SFT @ 3000 steps

Three SFT arms trained at 3000 steps each (3× longer than the previous Kaggle run), then evaluated against base on GSM8K + MATH via the exact-answer eval CLI. Designed to be uploaded into the pod's JupyterLab and run cell-by-cell.

**Why 3000 steps:** The 1000-step base-vs-combined run (`results/evals/base_vs_sft_run_2`) showed SFT had taught format (parse-success jumped 22% → 86% on GSM8K) but not capability (accuracy only +1.6 pp on GSM8K, **−15.8 pp on MATH**). Tripling the steps tests whether the model can actually learn math reasoning given more compute, or whether the recipe itself is the limit.

**Three arms (same as Kaggle ablation):**
- `gsm8k_only` — sources `[gsm8k]`, 3000 steps
- `math_only` — sources `[math]`, 3000 steps
- `combined` — sources `[gsm8k, math]`, 3000 steps

**Hyperparameters match the Kaggle ablation notebook** (`sft_phase1_kaggle_ablation_complete.ipynb`) — fp32, batch=1, grad_accum=4, lr=1e-5, max_seq_len=512 — so the comparison to your existing `combined` 1000-step checkpoint is clean on everything except step count.

**Time and cost estimate (A40 on-demand at $0.44/hr):**

| Step | Time | Cost |
|---|---|---|
| 3× SFT 3000 steps (~75 min each) | ~225 min | ~$1.65 |
| 3× checkpoint conversion | ~3 min | ~$0.02 |
| Eval (3 ckpts × 2 sources × 500) | ~22 min | ~$0.16 |
| **Total** | **~4.2 h** | **~$1.83** |

**Before running this notebook:** make sure you've `git pull`-ed in `/workspace/finpost` so the latest YAML conventions, `convert_checkpoint_to_hf.py`, and eval CLI are present.

## Step 1 — Sanity-check the pod

GPU should be A40 with ~48 GB free, CUDA 12.x. If you don't see that, terminate and re-deploy.

In [ ]:
!nvidia-smi

In [ ]:
import os
import subprocess
import sys

# The notebook expects to live next to (or be uploaded into) /workspace/finpost.
# Switch into the repo so all later commands resolve from the package root.
REPO_ROOT = '/workspace/finpost'
os.chdir(REPO_ROOT)
print('cwd:', os.getcwd())

# Make sure we're on the latest main so the convert script + YAML conventions
# match what this notebook assumes.
subprocess.run(['git', '-C', REPO_ROOT, 'pull'], check=False)
subprocess.run(['git', '-C', REPO_ROOT, 'log', '--oneline', '-1'], check=False)

# Activate the venv that the previous session set up. If this notebook is
# the first thing on a fresh pod, run pip install -e ".[dev]" in a terminal
# before continuing.
import torch
import finpost
print('\nfinpost:', finpost.__version__)
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

## Step 2 — Ablation constants

All three arms share these hyperparameters. The only thing that differs is `sources` (which dataset(s) the arm trains on). This is the same recipe the Kaggle notebook used, except `ABLATION_STEPS` is bumped from 1000 to 3000.

In [ ]:
# Training hyperparameters — match Kaggle ablation, except 3× longer.
ABLATION_STEPS = 3000
MAX_SEQ_LEN = 512
DTYPE = 'float32'
LR = 1.0e-5
PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 4
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
VAL_SPLIT_PCT = 5.0
SEED = 42

# Proportional warmup (10% of training). Validator requires warmup_steps < max_steps.
warmup_steps = max(50, ABLATION_STEPS // 10)

# Effective batch size = per_device_batch_size * grad_accum_steps.
effective_batch = PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS

# Eval settings (applied after all three arms train).
EVAL_N = 500
EVAL_SEED = 42
BATCH_SIZE_GSM8K = 16
BATCH_SIZE_MATH = 8
GPU_COST_PER_HOUR = 0.44

# Where artifacts go.
EXPERIMENTS_DIR = 'experiments/runpod_3000'
CHECKPOINTS_DIR = 'results/checkpoints'
EVAL_OUT_DIR = 'results/evals/ablation_3000_run_1'

print(f'ABLATION_STEPS: {ABLATION_STEPS}')
print(f'warmup_steps:   {warmup_steps}')
print(f'effective batch:{effective_batch}')
print(f'lr:             {LR}')
print(f'max_seq_len:    {MAX_SEQ_LEN}')
print(f'dtype:          {DTYPE}')

## Step 3 — Generate the three YAML configs

Written to `experiments/runpod_3000/` so they're reproducible and visible alongside the run.

In [ ]:
import yaml
from pathlib import Path

ARMS = {
    'gsm8k_only': ['gsm8k'],
    'math_only':  ['math'],
    'combined':   ['gsm8k', 'math'],
}

def build_config(arm_name: str, sources: list[str]) -> dict:
    """Build the trainer YAML for one arm. Hyperparameters from notebook constants."""
    run_name = f'qwen-{arm_name}-{ABLATION_STEPS}s'
    return {
        'model': {
            'base_model_id': 'Qwen/Qwen2.5-0.5B',
            'dtype': DTYPE,
            'use_safetensors': True,
        },
        'data': {
            'sources': sources,
            'val_split_pct': VAL_SPLIT_PCT,
            'seed': SEED,
        },
        'training': {
            'max_steps': ABLATION_STEPS,
            'warmup_steps': warmup_steps,
            'lr': LR,
            'weight_decay': WEIGHT_DECAY,
            'grad_accum_steps': GRAD_ACCUM_STEPS,
            'grad_clip': GRAD_CLIP,
            # Validate every 1/4 of training, plus the final.
            'val_every_n_steps': max(250, ABLATION_STEPS // 4),
            # Save only the final checkpoint (retention_last_n caps disk usage anyway).
            'checkpoint_every_n_steps': ABLATION_STEPS,
            'per_device_batch_size': PER_DEVICE_BATCH_SIZE,
        },
        'packing': {
            'max_seq_len': MAX_SEQ_LEN,
            'isolate_documents': True,
        },
        'logging': {
            'wandb_project': 'finpost-phase1-runpod',
            'run_name': run_name,
        },
        'checkpointing': {
            'save_dir': f'{CHECKPOINTS_DIR}/{run_name}',
            'retention_last_n': 1,
            'resume_from': None,
        },
    }

Path(EXPERIMENTS_DIR).mkdir(parents=True, exist_ok=True)

yaml_paths: dict[str, Path] = {}
for arm_name, sources in ARMS.items():
    cfg = build_config(arm_name, sources)
    yaml_path = Path(f'{EXPERIMENTS_DIR}/{arm_name}_{ABLATION_STEPS}.yaml')
    yaml_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
    yaml_paths[arm_name] = yaml_path
    print(f'wrote {yaml_path}')

# Sanity-check by printing one of them so you can scan the values.
print('\n--- combined config ---')
print(yaml_paths['combined'].read_text())

## Step 4 — Train each arm sequentially

Three separate cells. Run them in order — each one trains for ~75 min on A40. Loss curves go to wandb (offline) and are also visible in the cell output.

**If you need to stop mid-run:** Ctrl-C the cell. The trainer doesn't automatically save mid-run unless `checkpoint_every_n_steps < max_steps`. Currently set to save only at the end, so partial training is lost.

**WANDB_MODE=offline** is set once for the whole notebook process — no cloud-auth dance, wandb writes to `wandb/offline-run-*/` for later sync if you want.

In [ ]:
os.environ['WANDB_MODE'] = 'offline'
print('WANDB_MODE =', os.environ['WANDB_MODE'])

### Arm 1 — gsm8k_only (~75 min)

In [ ]:
!python -m finpost.training.train \
    --config {yaml_paths['gsm8k_only']} \
    --device cuda

### Arm 2 — math_only (~75 min)

In [ ]:
!python -m finpost.training.train \
    --config {yaml_paths['math_only']} \
    --device cuda

### Arm 3 — combined (~75 min)

In [ ]:
!python -m finpost.training.train \
    --config {yaml_paths['combined']} \
    --device cuda

## Step 5 — Verify all three checkpoints landed

In [ ]:
for arm_name in ARMS.keys():
    run_name = f'qwen-{arm_name}-{ABLATION_STEPS}s'
    final_step_dir = Path(f'{CHECKPOINTS_DIR}/{run_name}/step-{ABLATION_STEPS:08d}')
    exists = final_step_dir.exists()
    print(f'{arm_name:12} → {final_step_dir} {"✓" if exists else "MISSING"}')
    if exists:
        print('   contents:', sorted(p.name for p in final_step_dir.iterdir()))

## Step 6 — Convert each checkpoint to HF format

The trainer writes raw weights (`model.safetensors` + `state.pt`). The eval CLI needs HF format (`config.json` + tokenizer + `model.safetensors`). Each conversion takes ~1 min.

In [ ]:
hf_paths: dict[str, str] = {}

for arm_name in ARMS.keys():
    run_name = f'qwen-{arm_name}-{ABLATION_STEPS}s'
    src = f'{CHECKPOINTS_DIR}/{run_name}/step-{ABLATION_STEPS:08d}'
    dst = f'{CHECKPOINTS_DIR}/{run_name}-hf'
    hf_paths[arm_name] = dst
    print(f'\n=== Converting {arm_name} ===')
    !python scripts/convert_checkpoint_to_hf.py \
        --checkpoint-dir {src} \
        --out-dir {dst} \
        --dtype {DTYPE}

## Step 7 — Run eval across all three arms (~22 min)

One CLI call across the three new HF-format checkpoints on both sources. Same seed and n as the previous run so per-example details are comparable.

If you also want `base=Qwen/Qwen2.5-0.5B` in the same eval for one-glance comparison, add it to the `--checkpoints` line — but you can also use the `base` numbers from `results/evals/base_vs_sft_run_2/accuracy_summary.csv` since the eval is deterministic at this seed.

In [ ]:
checkpoint_args = ' '.join(f'{arm}={path}' for arm, path in hf_paths.items())
print('--checkpoints', checkpoint_args)
print()

!python -m finpost.evals.eval_exact \
    --checkpoints {checkpoint_args} \
    --sources gsm8k math \
    --n {EVAL_N} --seed {EVAL_SEED} \
    --out-dir {EVAL_OUT_DIR} \
    --batch-size-gsm8k {BATCH_SIZE_GSM8K} --batch-size-math {BATCH_SIZE_MATH} \
    --gpu-cost-per-hour {GPU_COST_PER_HOUR} \
    --device cuda

## Step 8 — Display the headline numbers

In [ ]:
import json
from pathlib import Path

summary_csv = Path(EVAL_OUT_DIR) / 'accuracy_summary.csv'
cost_json = Path(EVAL_OUT_DIR) / 'cost_summary.json'

print('=== accuracy_summary.csv ===')
print(summary_csv.read_text())

print('\n=== cost_summary.json ===')
print(cost_json.read_text())

In [ ]:
# Optional — also load the previous run's base numbers for an inline 4-arm table.
try:
    import pandas as pd
    df_new = pd.read_csv(summary_csv)
    print('\n=== Ablation @ 3000 steps ===')
    print(df_new.to_string(index=False))

    base_csv = Path('results/evals/base_vs_sft_run_2/accuracy_summary.csv')
    if base_csv.exists():
        df_base = pd.read_csv(base_csv)
        df_base = df_base[df_base['checkpoint'] == 'base']
        df_all = pd.concat([df_base, df_new], ignore_index=True)
        print('\n=== With base (from base_vs_sft_run_2) ===')
        print(df_all[['checkpoint', 'source', 'n', 'accuracy', 'parse_success_rate']].to_string(index=False))
except ImportError:
    print('pandas not available — read the CSV manually above.')

## Step 9 — Tar the eval results for download

Bundles the 10-file eval output (3 ckpts × 2 sources × details CSV + 4 summary files) into a single tarball. Then in JupyterLab's file browser, navigate to `results/evals/`, right-click `ablation_3000_run_1.tar.gz`, and click **Download**.

In [ ]:
!tar -czf {EVAL_OUT_DIR}.tar.gz -C results/evals ablation_3000_run_1
!ls -lah {EVAL_OUT_DIR}.tar.gz

## Step 10 — (Optional) push the three trained checkpoints to HF Hub

Useful if you want to re-eval them later from a different pod without retraining. Each is ~2 GB. Skip this if you don't plan to come back to these specific checkpoints — the volume disk on a stopped pod keeps them around for ~$0.14/day anyway.

In [ ]:
# Uncomment to push. Requires `hf auth login` with a Write token first.
# for arm_name, hf_dir in hf_paths.items():
#     repo_id = f'sl891/finpost-sft-{arm_name}-{ABLATION_STEPS}'
#     print(f'\n=== Uploading {arm_name} → {repo_id} ===')
#     !hf upload {repo_id} {hf_dir}

## Final — Stop the pod

Download the tarball via the JupyterLab file browser, then in the RunPod console:

- **Stop Pod** — billing stops, volume persists (~$0.14/day idle). Use this if you'll be back within a few days.
- **Terminate Pod** — frees everything, including the venv and checkpoints. Use this when fully done with this experiment.

**Total spend after this run:** previous session (~$0.85) + this ablation (~$1.85) = **~$2.70** so far.